In [1]:
from ioMicro import *

In [2]:
data_folder = r'Z:\Zane_20CRE\7_17_2024__T7_20CRE'

In [3]:
# map all the hybes
hybes =  glob.glob(data_folder+os.sep+'H*')
# map all the fovs
fovs = [os.path.basename(fl)for fl in glob.glob(hybes[0]+os.sep+'*.zarr')]
def get_Hi(fld): 
    try: return int(os.path.basename(fld)[1:]) 
    except: return -1
    
hybes = np.array(hybes)[np.argsort([get_Hi(hybe) for hybe in hybes ])]

In [4]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

In [15]:
from scipy.signal import fftconvolve

def fit(hybe,fov,analysis_folder='/projects/ps-renlab2/zgibbs/STARR-FISH/10_12_2023__CRE-20_RNASEQ',icol=0,redo=False,hth = 50,plt_val=True):
    fl = hybe+os.sep+fov
    tag = os.path.basename(hybe)
    save_file=analysis_folder+os.sep+fov.split('.')[0]+'--'+tag+'_fits_icol'+str(icol)+'.npz'
    if not os.path.exists(save_file) or redo:
        im = read_im(fl)
        ncols,sz,sx,sy = im.shape
        #for icol in np.arange(ncols-1):
        imS = np.array(im[icol],dtype=np.float32) #GFP cy5 half
        #Convolve with gaussian 
        sz=5
        X = np.indices([2*sz+1]*3)-sz
        sigma=1.5
        gaussian = np.exp(-np.sum(X**2,axis=0)/2/sigma**2)
        im_conv = fftconvolve(imS,gaussian,mode='same')/np.sum(gaussian)
        #Subtract local background
        im_conv_ = norm_slice(im_conv,s=30)
        #fit the spots
        Xh = get_local_max(im_conv_,hth,im_raw=im_conv,delta=1,delta_fit=3)
        if plt_val:
            import napari
            v = napari.view_image(im_conv_)
            v.add_image(imS)
            h = Xh[:,-1]
            size = 5+np.clip(h/np.percentile(h,99.9),0,1)*10
            v.add_points(Xh[:,:3],size=size,face_color=[0,0,0,0],edge_color='y')
        np.savez_compressed(save_file,Xh=Xh)

In [16]:
def tag_to_hybe(tag):
    try:
        return int(tag[1:])
    except:
        return -1

def get_XH(save_folder='something', fov='something', ncols=3, chromatic_fl='something', drift_fl='something'):
        """Load in the fitted dots for each field of view.
        Apply chromatic abberation correction and drift correction and append corrected fitted data 
        int self.XH structure"""
        ncols = ncols
        drift_dic = pickle.load(open(drift_fl,'rb'))
        tags = list(drift_dic.keys())
        if not os.path.exists(chromatic_fl):
            self.m=None
        else:
            m = np.load(chromatic_fl)
        XH = []
        save_folder = save_folder
        fov = fov
        for tag in tqdm(tags):
            iH = tag_to_hybe(tag)
            for icol in range(ncols):
                #tag = os.path.basename(fld)#Conv_zscan__30--H_GFP_fits_icol0
                save_fl = save_folder+os.sep+fov.split('.')[0]+'--'+tag+'_fits_icol'+str(icol)+'.npz'
                Xh = np.load(save_fl)['Xh']
                tzxy = drift_dic[tag][0]
                m=None
                if icol==0: m=m
                Xh[:,:3] = apply_colorcor(Xh[:,:3],m=m)
                Xh[:,:3]+=tzxy# drift correction
                #chromatic abberation -to check
                ### took out code for assigning bits to array ###
        XH = np.array(Xh)

In [28]:
def main_do_compute_fits(save_folder,hybe,fov,icol,save_fl,psf,old_method):
    im_ = read_im(hybe+os.sep+fov)
    im__ = np.array(im_[icol],dtype=np.float32)
    
    if old_method:
        ### previous method
        im_n = norm_slice(im__,s=30)
        #Xh = get_local_max(im_n,500,im_raw=im__,dic_psf=None,delta=1,delta_fit=3,dbscan=True,
        #      return_centers=False,mins=None,sigmaZ=1,sigmaXY=1.5)
        Xh = get_local_maxfast_tensor(im_n,th_fit=500,im_raw=im__,dic_psf=None,delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5,gpu=True)
    else:
        ### new method
        fl_med = flat_field_tag+'med_col_raw'+str(icol)+'.npz'
        if os.path.exists(fl_med):
            im_med = np.array(np.load(fl_med)['im'],dtype=np.float32)
            im_med = cv2.blur(im_med,(20,20))
            im__ = im__/im_med*np.median(im_med)
        else:
            print("Did not find flat field")
        try:
            Xh = get_local_max_tile(im__,th=3600,s_ = 500,pad=100,psf=psf,plt_val=None,snorm=30,gpu=True,
                                    deconv={'method':'wiener','beta':0.0001},
                                    delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5)
        except:
            Xh = get_local_max_tile(im__,th=3600,s_ = 500,pad=100,psf=psf,plt_val=None,snorm=30,gpu=False,
                                    deconv={'method':'wiener','beta':0.0001},
                                    delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5)
    np.savez_compressed(save_fl,Xh=Xh)

In [43]:
def get_local_maxfast_tensor(m,th_fit=500,im_raw=None,dic_psf=None,delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5,gpu=False):
    import torch
    dev = "cuda:0" if (torch.cuda.is_available() and gpu) else "cpu"
    im_dif = torch.from_numpy(m).to(dev)
    z,x,y = torch.where(im_dif>th_fit)
    zmax,xmax,ymax = im_dif.shape
    def get_ind(x,xmax):
        # modify x_ to be within image
        x_ = torch.clone(x)
        bad = x_>=xmax
        x_[bad]=xmax-x_[bad]-2
        bad = x_<0
        x_[bad]=-x_[bad]
        return x_
    #def get_ind(x,xmax):return x%xmax
    for d1 in range(-delta,delta+1):
        for d2 in range(-delta,delta+1):
            for d3 in range(-delta,delta+1):
                if (d1*d1+d2*d2+d3*d3)<=(delta*delta):
                    z_ = get_ind(z+d1,zmax)
                    x_ = get_ind(x+d2,xmax)
                    y_ = get_ind(y+d3,ymax)
                    keep = im_dif[z,x,y]>=im_dif[z_,x_,y_]
                    z,x,y = z[keep],x[keep],y[keep]
    h = im_dif[z,x,y]
    
    
    if len(x)==0:
        return []
    if delta_fit>0:
        d1,d2,d3 = np.indices([2*delta_fit+1]*3).reshape([3,-1])-delta_fit
        kp = (d1*d1+d2*d2+d3*d3)<=(delta_fit*delta_fit)
        d1,d2,d3 = d1[kp],d2[kp],d3[kp]
        d1 = torch.from_numpy(d1).to(dev)
        d2 = torch.from_numpy(d2).to(dev)
        d3 = torch.from_numpy(d3).to(dev)
        im_centers0 = (z.reshape(-1, 1)+d1).T
        im_centers1 = (x.reshape(-1, 1)+d2).T
        im_centers2 = (y.reshape(-1, 1)+d3).T
        z_ = get_ind(im_centers0,zmax)
        x_ = get_ind(im_centers1,xmax)
        y_ = get_ind(im_centers2,ymax)
        im_centers3 = im_dif[z_,x_,y_]
        if im_raw is not None:
            im_raw_ = torch.from_numpy(im_raw).to(dev)
            im_centers4 = im_raw_[z_,x_,y_]
            habs = im_raw_[z,x,y]
        else:
            im_centers4 = im_dif[z_,x_,y_]
            habs = x*0
            a = x*0
        Xft = torch.stack([d1,d2,d3]).T
    
        bk = torch.min(im_centers3,0).values
        im_centers3 = im_centers3-bk
        im_centers3 = im_centers3/torch.sum(im_centers3,0)
        if dic_psf is None:
            sigma = torch.tensor([sigmaZ,sigmaXY,sigmaXY],dtype=torch.float32,device=dev)#np.array([sigmaZ,sigmaXY,sigmaXY],dtype=np.flaot32)[np.newaxis]
            Xft_ = Xft/sigma
            norm_G = torch.exp(-torch.sum(Xft_*Xft_,-1)/2.)
            norm_G=(norm_G-torch.mean(norm_G))/torch.std(norm_G)
    
            hn = torch.mean(((im_centers3-im_centers3.mean(0))/im_centers3.std(0))*norm_G.reshape(-1,1),0)
            a = torch.mean(((im_centers4-im_centers4.mean(0))/im_centers4.std(0))*norm_G.reshape(-1,1),0)
            
        zc = torch.sum(im_centers0*im_centers3,0)
        xc = torch.sum(im_centers1*im_centers3,0)
        yc = torch.sum(im_centers2*im_centers3,0)
        Xh = torch.stack([zc,xc,yc,bk,a,habs,hn,h]).T.cpu().detach().numpy()
    else:
        Xh =  torch.stack([z,x,y,h]).T.cpu().detach().numpy()
    return Xh

In [19]:
hybes[1:]

array(['Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H0',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H1',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H2',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H3',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H4',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H5',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H6',
       'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H7'], dtype='<U38')

In [ ]:
save_folder = r'Z:\Zane_20CRE\7_17_2024__T7_20CRE\enhancer_fits'
fld =  r'Z:\Zane_20CRE\7_17_2024__T7_20CRE'
tag = os.path.basename(hybe)
save_fl = fld+os.sep+fov.split('.')[0]+'--'+tag+'_fits_icol'+str(icol)+'.npz'
psf = r'C:\Scripts\NMERFISH\psfs\psf_750_Scope2_Tasic_final.npy'

for fov in tqdm(fovs):
    for hybe in hybes[1:]:
        for icol in range(3):
            main_do_compute_fits(save_folder,hybe,fov,icol,save_fl,psf,old_method=True)

  0%|▋                                                                                                                                                           | 1/225 [26:06<97:26:50, 1566.12s/it]

In [5]:
def get_local_max_tile(im_,th=2500,s_ = 300,pad=50,psf=None,plt_val=None,snorm=30,gpu=False,deconv={'method':'wiener','beta':0.001},
                        delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5):
    sx,sy = im_.shape[1:]
    ixys = []
    for ix in np.arange(0,sx,s_):
        for iy in np.arange(0,sy,s_):
            ixys.append([ix,iy])
    Xhf = None
    for ix,iy in ixys:#tqdm(ixys):
        imsm = im_[:,ix:ix+pad+s_,iy:iy+pad+s_]
        out_im = imsm
        if deconv is not None:
            force = True
            psf_ = psf
            if type(psf) is dict:
                force=True
                keys = list(psf.keys())
                ikey = np.argmin(np.sum(np.abs(np.array(keys)-[0,ix,iy]),axis=-1))
                psf_ = psf[keys[ikey]]
            out_im = apply_deconv(imsm,psf=psf_,plt_val=False,parameters = deconv,gpu=gpu,force=True,pad=None)
        out_im2 = norm_slice(out_im,s=snorm)
        #print(time.time()-t)
        Xh = get_local_maxfast_tensor(out_im2,th,im_raw=imsm,dic_psf=None,delta=delta,delta_fit=delta_fit,sigmaZ=sigmaZ,sigmaXY=sigmaXY,gpu=gpu)
        ### exclude outside the padded area
        if Xh is not None:
            if len(Xh)>0:
                keep = np.all(Xh[:,1:3]<(s_+pad/2),axis=-1)
                keep &= np.all(Xh[:,1:3]>=(pad/2*np.array([ix>0,iy>0])),axis=-1)
                Xh = Xh[keep]
                Xh[:,1]+=ix
                Xh[:,2]+=iy
                #Xh[:,:3]-=1
                if Xhf is None: Xhf=Xh
                else: Xhf = np.concatenate([Xhf,Xh])
        #print(time.time()-t)
    if plt_val is not None:
        import napari
        v = napari.Viewer()
        #im__ = norm_slice(im_,s=30)
        v.add_image(im_,name='Original image')
        v.add_image(out_im2,name='Deconv image')
        H= Xhf[:,-1]
        size=None
        if type(plt_val) is dict:
            size = plt_val.get('size')
        if size is None:
            size = np.clip(H/np.percentile(H,99.99),0,1)*5
        v.add_points(Xhf[:,:3],face_color=[0,0,0,0],edge_color='y',size=size)
    return Xhf

In [6]:
def get_local_maxfast_tensor(m,th_fit=500,im_raw=None,dic_psf=None,delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5,gpu=False):
    import torch
    dev = "cuda:0" if (torch.cuda.is_available() and gpu) else "cpu"
    im_dif = torch.from_numpy(im_dif_npy).to(dev)
    z,x,y = torch.where(im_dif>th_fit)
    zmax,xmax,ymax = im_dif.shape
    def get_ind(x,xmax):
        # modify x_ to be within image
        x_ = torch.clone(x)
        bad = x_>=xmax
        x_[bad]=xmax-x_[bad]-2
        bad = x_<0
        x_[bad]=-x_[bad]
        return x_
    #def get_ind(x,xmax):return x%xmax
    for d1 in range(-delta,delta+1):
        for d2 in range(-delta,delta+1):
            for d3 in range(-delta,delta+1):
                if (d1*d1+d2*d2+d3*d3)<=(delta*delta):
                    z_ = get_ind(z+d1,zmax)
                    x_ = get_ind(x+d2,xmax)
                    y_ = get_ind(y+d3,ymax)
                    keep = im_dif[z,x,y]>=im_dif[z_,x_,y_]
                    z,x,y = z[keep],x[keep],y[keep]
    h = im_dif[z,x,y]
    
    
    if len(x)==0:
        return []
    if delta_fit>0:
        d1,d2,d3 = np.indices([2*delta_fit+1]*3).reshape([3,-1])-delta_fit
        kp = (d1*d1+d2*d2+d3*d3)<=(delta_fit*delta_fit)
        d1,d2,d3 = d1[kp],d2[kp],d3[kp]
        d1 = torch.from_numpy(d1).to(dev)
        d2 = torch.from_numpy(d2).to(dev)
        d3 = torch.from_numpy(d3).to(dev)
        im_centers0 = (z.reshape(-1, 1)+d1).T
        im_centers1 = (x.reshape(-1, 1)+d2).T
        im_centers2 = (y.reshape(-1, 1)+d3).T
        z_ = get_ind(im_centers0,zmax)
        x_ = get_ind(im_centers1,xmax)
        y_ = get_ind(im_centers2,ymax)
        im_centers3 = im_dif[z_,x_,y_]
        if im_raw is not None:
            im_raw_ = torch.from_numpy(im_raw).to(dev)
            im_centers4 = im_raw_[z_,x_,y_]
            habs = im_raw_[z,x,y]
        else:
            im_centers4 = im_dif[z_,x_,y_]
            habs = x*0
            a = x*0
        Xft = torch.stack([d1,d2,d3]).T
    
        bk = torch.min(im_centers3,0).values
        im_centers3 = im_centers3-bk
        im_centers3 = im_centers3/torch.sum(im_centers3,0)
        if dic_psf is None:
            sigma = torch.tensor([sigmaZ,sigmaXY,sigmaXY],dtype=torch.float32,device=dev)#np.array([sigmaZ,sigmaXY,sigmaXY],dtype=np.flaot32)[np.newaxis]
            Xft_ = Xft/sigma
            norm_G = torch.exp(-torch.sum(Xft_*Xft_,-1)/2.)
            norm_G=(norm_G-torch.mean(norm_G))/torch.std(norm_G)
    
            hn = torch.mean(((im_centers3-im_centers3.mean(0))/im_centers3.std(0))*norm_G.reshape(-1,1),0)
            a = torch.mean(((im_centers4-im_centers4.mean(0))/im_centers4.std(0))*norm_G.reshape(-1,1),0)
            
        zc = torch.sum(im_centers0*im_centers3,0)
        xc = torch.sum(im_centers1*im_centers3,0)
        yc = torch.sum(im_centers2*im_centers3,0)
        Xh = torch.stack([zc,xc,yc,bk,a,habs,hn,h]).T.cpu().detach().numpy()
    else:
        Xh =  torch.stack([z,x,y,h]).T.cpu().detach().numpy()
    return Xh

In [15]:
hybe+os.sep+fov

'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H0\\Conv_zscan__000.zarr'

In [ ]:
im_dif_npy = im__

In [7]:
save_folder = r'Z:\Zane_20CRE\7_17_2024__T7_20CRE\enhancer_fits'
fld =  r'Z:\Zane_20CRE\7_17_2024__T7_20CRE'
# tag = os.path.basename(hybe)
# save_fl = fld+os.sep+fov.split('.')[0]+'--'+tag+'_fits_icol'+str(icol)+'.npz'
psf = r'C:\Scripts\NMERFISH\psfs\psf_750_Scope2_Tasic_final.npy'

for fov in tqdm(fovs):
    for hybe in hybes[1:]:
        for icol in range(3):
            im_ = read_im(hybe+os.sep+fov)
            im__ = np.array(im_[icol],dtype=np.float32)
            im_dif_npy = im__
            get_local_max_tile(im__,th=2500,s_ = 300,pad=50,psf=None,plt_val=None,snorm=30,gpu=True,deconv={'method':'wiener','beta':0.001},
                        delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5)

  0%|                                                                                                                                                                       | 0/225 [00:22<?, ?it/s]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [7]:
!pip install sdeconv

  Using cached sdeconv-1.0.2-py3-none-any.whl.metadata (3.3 kB)
Using cached sdeconv-1.0.2-py3-none-any.whl (34.0 MB)


In [20]:
x, y = im__.shape[1:]

In [21]:
x

3000

In [7]:
import torch

In [9]:
torch.cuda.is_available()

True

In [8]:
torch.cuda.get_device_name(0)

'NVIDIA RTX A6000'

In [27]:
hybe

'Z:\\Zane_20CRE\\7_17_2024__T7_20CRE\\H0'

In [36]:
?get_local_maxfast_tensor

Signature:
get_local_maxfast_tensor(
    m,
    th_fit=500,
    im_raw=None,
    dic_psf=None,
    delta=1,
    delta_fit=3,
    sigmaZ=1,
    sigmaXY=1.5,
    gpu=False,
)
Docstring: <no docstring>
File:      c:\users\zgibbs\appdata\local\temp\ipykernel_12180\2199543467.py
Type:      function

In [37]:
fov

'Conv_zscan__000.zarr'